# Neural Network (NN1) on MALL Feature Set

**Purpose:** This notebook implements the shallow neural network architecture NN1 from Christensen, Siggaard, and Veliyev (2023) on the MALL feature set for the EURO STOXX 50 index. The network has a single hidden layer with two neurons and serves as the simplest neural-network benchmark considered in the paper. The implementation below follows the MHAR NN1 notebook as closely as possible, while documenting the same practical departures made in this thesis: (i) the ensemble size and training budget are reduced because the full paper configuration is computationally expensive in this environment, and (ii) the model is re-estimated every 22 trading days in a rolling scheme, whereas the paper uses a fixed-window neural-network estimation. The only substantive change relative to NN1_MHAR is that the predictors now come from the compact main-thesis MALL feature set.

## 1. Imports

We import the libraries required for data handling, numerical computation, and neural-network estimation. TensorFlow and Keras are used to define and train the NN1 architecture.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')


## 2. Data Loading

We use the final `MALL.csv` data source so the neural-network model is evaluated on the same target and information set as the other MALL notebooks. The file itself is left unchanged; the compact main-thesis MALL feature set is defined within the notebook.

In [ ]:
MALL = pd.read_csv('/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv')
MALL['Date'] = pd.to_datetime(MALL['Date'])
MALL = MALL.set_index('Date')

print(f'Sample period: {MALL.index[0].date()} to {MALL.index[-1].date()}')
print(f'Number of observations: {len(MALL)}')
display(MALL[['RVD', 'RVW', 'RVM', 'VSTOXX', 'CISS', 'RV_target']].head())


## 3. Data Split

To remain directly comparable with the other MALL notebooks, we use the same 70/10/20 split into training, validation, and test sets. The validation block is used for early stopping and ensemble selection.

In [ ]:
n = len(MALL)
n_train = int(np.floor(0.70 * n))
n_val = int(np.floor(0.10 * n))
n_test = n - n_train - n_val

print(f'Total observations: {n}')
print(f'\nTraining:   {MALL.index[0].date()} to {MALL.index[n_train-1].date()} ({n_train} obs)')
print(f'Validation: {MALL.index[n_train].date()} to {MALL.index[n_train+n_val-1].date()} ({n_val} obs)')
print(f'Test:       {MALL.index[n_train+n_val].date()} to {MALL.index[-1].date()} ({n_test} obs)')


## 4. Feature Set Construction and Standardization

The MALL specification uses the compact main-thesis MALL feature set rather than the full collection of variables in `MALL.csv`. This keeps the model focused on the most relevant and interpretable drivers of European equity-market volatility while avoiding redundant transforms. Following the paper and the MHAR NN1 notebook, both predictors and the target are standardized using moments from the training set only. Standardizing the target is convenient in neural-network training because it stabilizes optimization and makes the scale of the loss function easier to manage.

In [ ]:
y = MALL['RV_target'].values
feature_cols = ['RVD', 'RVW', 'RVM', 'RD_neg', 'RW_neg', 'RM_neg', 'RQ', 'VSTOXX', 'CISS', 'logEPU_1m_lag', 'EURUSD_ret', 'BRENT_ret', 'SP500_ret_lag', 'SP500_sq_lag', 'M1W', 'M1M']
X_MALL = MALL[feature_cols].values

X_train = X_MALL[:n_train]
X_val = X_MALL[n_train:n_train + n_val]
X_test = X_MALL[n_train + n_val:]

y_train = y[:n_train]
y_val = y[n_train:n_train + n_val]
y_test = y[n_train + n_val:]

X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)
y_mean = y_train.mean()
y_std = y_train.std()

X_scaled = (X_MALL - X_mean) / X_std
y_scaled = (y - y_mean) / y_std

print('Feature set: RVD, RVW, RVM, RD_neg, RW_neg, RM_neg, RQ, VSTOXX, CISS, logEPU_1m_lag, EURUSD_ret, BRENT_ret, SP500_ret_lag, SP500_sq_lag, M1W, M1M')
print(f'\nNumber of predictors: {len(feature_cols)}')
print(f'Full dataset:   X={X_scaled.shape}, y={y_scaled.shape}')
print(f'Training set:   X={X_scaled[:n_train].shape}, y={y_scaled[:n_train].shape}')
print(f'Validation set: X={X_scaled[n_train:n_train+n_val].shape}, y={y_scaled[n_train:n_train+n_val].shape}')
print(f'Test set:       X={X_scaled[n_train+n_val:].shape}, y={y_scaled[n_train+n_val:].shape}')
print('\nStandardization check on training sample:')
print(f'  X means: {X_scaled[:n_train].mean(axis=0).round(6)}')
print(f'  X stds:  {X_scaled[:n_train].std(axis=0).round(6)}')
print(f'  y mean:  {y_scaled[:n_train].mean():.6f}')
print(f'  y std:   {y_scaled[:n_train].std():.6f}')
print('\nPredictor order:')
for j, col in enumerate(feature_cols, start=1):
    print(f'  {j:>2d}. {col}')


## 5. NN1 Architecture

The paper's NN1 architecture is a shallow feed-forward network with a single hidden layer containing two neurons. We follow the paper in using Leaky ReLU activation with slope parameter `0.01`, Adam with learning rate `0.001`, Glorot normal initialization, and dropout with keep rate `0.8` (implemented as dropout rate `0.2`). The only architecture-level change relative to NN1_MHAR is that the input dimension now matches the MALL feature count.

In [ ]:
def build_NN1(input_dim, seed=None):
    if seed is not None:
        tf.random.set_seed(seed)

    initializer = keras.initializers.GlorotNormal(seed=seed)

    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(2, kernel_initializer=initializer, bias_initializer='zeros'),
        layers.LeakyReLU(negative_slope=0.01),
        layers.Dropout(rate=0.2),
        layers.Dense(1, kernel_initializer=initializer, bias_initializer='zeros', activation='linear')
    ])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

test_model = build_NN1(input_dim=X_scaled.shape[1], seed=0)
test_model.summary()
print(f'\nInput dimension in this notebook: {X_scaled.shape[1]} predictors')


## 6. Hyperparameters and Departures from the Paper

This section is where the thesis implementation departs most clearly from the reference paper. The paper trains `100` independent networks and averages the best `10`, uses up to `500` epochs, patience `100`, and applies a fixed-window neural-network estimation. In this thesis, we reduce the computational burden by using a smaller ensemble, fewer epochs, and a monthly rolling re-estimation frequency of `22` trading days. The latter is an extension relative to the paper: it allows the model to adapt over time, but it is not the exact estimation design used in the published study. These departures are the same as in NN1_MHAR; the MALL specification changes only the feature set.

In [ ]:
n_ensemble = 10
n_best = 3
max_epochs = 200
patience = 50
refit_freq = 22

print('Hyperparameter summary:')
print(f'  Ensemble size:      {n_ensemble} (paper: 100)')
print(f'  Best networks kept: {n_best} (paper: 10)')
print(f'  Max epochs:         {max_epochs} (paper: 500)')
print(f'  Patience:           {patience} (paper: 100)')
print(f'  Refit frequency:    every {refit_freq} trading days (paper: fixed window)')
print('  Learning rate:      0.001 (same as paper)')
print('  Dropout keep rate:  0.8 (same as paper)')
print('  Activation:         Leaky ReLU with slope 0.01 (same as paper)')
print('  Initializer:        Glorot normal (same as paper)')


## 7. Rolling Window Estimation and Forecasting

The rolling forecast design keeps the train-plus-validation window length fixed and moves it through the test sample. Every `22` trading days, we retrain the ensemble on the current window, rank the networks by validation loss, retain the best-performing subset, and use their average prediction as the one-step-ahead forecast. This is a deliberate departure from the paper's fixed-window neural-network setup and is adopted to let the network incorporate more recent information without retraining every single day.

In [ ]:
window_size = n_train + n_val
forecasts_NN1 = []
dates_test = []
best_models = None
last_refit = -refit_freq
n_refits = int(np.ceil(n_test / refit_freq))

start_time = time.time()

for i in range(n_test):
    if i - last_refit >= refit_freq:
        X_window = X_scaled[i:i + window_size]
        y_window = y_scaled[i:i + window_size]

        X_win_train = X_window[:-n_val]
        X_win_val = X_window[-n_val:]
        y_win_train = y_window[:-n_val]
        y_win_val = y_window[-n_val:]

        val_losses_window = []
        models_window = []

        for j in range(n_ensemble):
            early_stopping = EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
                verbose=0
            )

            model = build_NN1(input_dim=X_scaled.shape[1], seed=j)
            model.fit(
                X_win_train, y_win_train,
                validation_data=(X_win_val, y_win_val),
                epochs=max_epochs,
                batch_size=len(X_win_train),
                callbacks=[early_stopping],
                verbose=0
            )
            val_loss = model.evaluate(X_win_val, y_win_val, verbose=0)
            val_losses_window.append(val_loss)
            models_window.append(model)

        top_idx = np.argsort(val_losses_window)[:n_best]
        best_models = [models_window[j] for j in top_idx]
        last_refit = i

        elapsed = (time.time() - start_time) / 60
        refit_num = i // refit_freq + 1
        print(f'  Refit {refit_num:3d}/{n_refits} | Test day {i:4d}/{n_test} | Date: {MALL.index[i + window_size].date()} | Best val MSE: {min(val_losses_window):.4f} | Elapsed: {elapsed:.1f} min')

    X_forecast = X_scaled[i + window_size].reshape(1, -1)
    preds_scaled = np.array([model.predict(X_forecast, verbose=0).flatten()[0] for model in best_models])
    forecast = preds_scaled.mean() * y_std + y_mean
    min_rv = y[i:i + window_size].min()
    forecast = max(forecast, min_rv)

    forecasts_NN1.append(forecast)
    dates_test.append(MALL.index[i + window_size])

total_elapsed = (time.time() - start_time) / 60
print(f'\nTotal estimation time: {total_elapsed:.1f} minutes')
print(f'Forecasts generated: {len(forecasts_NN1)}')
print(f'Forecast period: {dates_test[0].date()} to {dates_test[-1].date()}')


## 8. Save Forecasts

We save the forecast series and the standard level-scale accuracy metrics in the same location and format as the rest of the MALL notebooks so that later comparison notebooks can load NN1 directly.

In [ ]:
forecast_path = '/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MALL'
os.makedirs(forecast_path, exist_ok=True)

test_dates = MALL.index[n_train + n_val:]
results_NN1 = pd.DataFrame({'RV_actual': y_test, 'RV_forecast': forecasts_NN1}, index=test_dates)
errors_NN1 = results_NN1['RV_actual'] - results_NN1['RV_forecast']
mse_NN1 = np.mean(errors_NN1 ** 2)

results_NN1.to_csv(os.path.join(forecast_path, 'forecasts_NN1.csv'))
mse_summary_NN1 = pd.DataFrame({
    'Model': ['NN1'],
    'MSE': [mse_NN1],
    'RMSE': [np.sqrt(mse_NN1)],
    'MAE': [errors_NN1.abs().mean()],
    'Mean_Error': [errors_NN1.mean()]
}).set_index('Model')
mse_summary_NN1.to_csv(os.path.join(forecast_path, 'mse_summary_NN1.csv'))

print(f'Forecasts saved to: {forecast_path}')
print(f'Rows saved: {len(results_NN1)}')
print(f'NN1 test-set MSE: {mse_NN1:.6e}')


## 9. Forecast Visualization

As in the other benchmark notebooks, we convert realized variance into annualized volatility before plotting the actual and forecasted series. This keeps the visual presentation intuitive and consistent across model classes, while leaving the model-comparison exercise to a later combined analysis.

In [ ]:
results_plot_NN1 = pd.DataFrame({
    'Actual': np.sqrt(results_NN1['RV_actual'] * 252) * 100,
    'Forecast': np.sqrt(results_NN1['RV_forecast'] * 252) * 100
}, index=results_NN1.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_plot_NN1['Actual'], label='Actual RV', color='black', linewidth=0.8)
ax.plot(results_plot_NN1['Forecast'], label='NN1 Forecast', color='red', linewidth=0.8, linestyle='--')
ax.set_title('Neural Network NN1 - One-Day-Ahead Realized Volatility Forecast\nEURO STOXX 50 (2016-2020)', fontsize=12)
ax.set_ylabel('Annualized Volatility (%)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Summary of forecast errors:')
print(f'Mean error:  {errors_NN1.mean():.6e}')
print(f'RMSE:        {np.sqrt(mse_NN1):.6e}')
print(f'MAE:         {errors_NN1.abs().mean():.6e}')


## 10. Departure Summary

To close the notebook, we summarize the differences between this thesis implementation and the exact design in the reference paper. The architecture and the core regularization devices follow the paper closely, but the estimation strategy is lighter and more adaptive because of computational constraints and the desire to let the network incorporate more recent data during the test period.

In [ ]:
departure_summary = pd.DataFrame({
    'Component': [
        'Architecture',
        'Feature set',
        'Learning rate',
        'Dropout keep rate',
        'Initializer',
        'Ensemble size',
        'Networks averaged',
        'Max epochs',
        'Patience',
        'Estimation window'
    ],
    'Paper': [
        '1 hidden layer, 2 neurons',
        'Broader M-ALL feature set',
        '0.001',
        '0.8',
        'Glorot normal',
        '100',
        '10 best',
        '500',
        '100',
        'Fixed window'
    ],
    'This thesis': [
        '1 hidden layer, 2 neurons',
        'Compact main-thesis MALL feature set',
        '0.001',
        '0.8',
        'Glorot normal',
        str(n_ensemble),
        str(n_best),
        str(max_epochs),
        str(patience),
        '22-day rolling refit'
    ]
})

display(departure_summary)
